# 02 Agent Architecture & Execution Loop

**Scenario**: Trace the conceptual lifecycle `Observe -> Reason -> Plan -> Tool -> Observe -> Reflect -> Repeat` using local Ollama `llama3.1:latest`.

This notebook implements a pure Python execution loop querying local Ollama to dynamically decide reasoning steps.

## Step 1: Inferences Setup & HTTP Connection

In [1]:
import urllib.request
import json

def query_local_llm(prompt, model="llama3.1:latest"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf-8"))
            return res.get("response", "").strip()
    except Exception as e:
        return f"[Fallback] Dynamic thought progression for prompt: {prompt[:40]}"

print("LLM Warmup:", query_local_llm("Explain execution loop in 3 words."))

LLM Warmup: Continuous process repeat.


## Step 2: Observe-Reason-Plan-Act Execution Loop

In [2]:
class RealExecutionLoopAgent:
    def __init__(self, goal):
        self.goal = goal
        self.state = {
            "step": 1,
            "observation": "Goal ingestion complete. Agent initialized.",
            "completed": False,
            "task_queue": ["Fetch status database", "Run memory diagnostics"]
        }
        
    def run_step(self):
        print(f"\n--- [STEP {self.state['step']}] ---")
        # 1. OBSERVE
        print(f"[OBSERVE] Last Observation: {self.state['observation']}")
        
        # 2. REASON
        reason_prompt = f"""Goal: {self.goal}
Current Task Queue: {self.state['task_queue']}
Last Observation: {self.state['observation']}
Reason in one sentence: What is the significance of the next task?"""
        thought = query_local_llm(reason_prompt)
        print(f"[REASON] Thought: {thought}")
        
        # 3. PLAN
        print(f"[PLAN] Remaining Tasks: {self.state['task_queue']}")
        
        # 4. ACT / TOOL
        if self.state["task_queue"]:
            active_tool = self.state["task_queue"].pop(0)
            print(f"[TOOL] Invoking tool for: [{active_tool}]")
            # Simulate environment execution
            self.state["observation"] = f"Tool [{active_tool}] execution successfully verified."
        else:
            print(f"[TOOL] No more active tasks in queue.")
            self.state["completed"] = True
            
        # 5. REFLECT
        reflect_prompt = f"""Goal: {self.goal}
Observation: {self.state['observation']}
Task Queue: {self.state['task_queue']}
Reflect in one sentence: Has the goal been resolved or is another step required?"""
        reflection = query_local_llm(reflect_prompt)
        print(f"[REFLECT] Reflection: {reflection}")
        
        self.state["step"] += 1
        if not self.state["task_queue"]:
            self.state["completed"] = True
            
    def execute(self):
        print(f"Starting Agent Execution Loop for Goal: '{self.goal}'")
        while not self.state["completed"] and self.state["step"] < 5:
            self.run_step()
            
agent = RealExecutionLoopAgent("Diagnose memory leakage anomalies")
agent.execute()

Starting Agent Execution Loop for Goal: 'Diagnose memory leakage anomalies'

--- [STEP 1] ---
[OBSERVE] Last Observation: Goal ingestion complete. Agent initialized.


[REASON] Thought: The significance of the next task "Run memory diagnostics" is that it will analyze the system's memory usage to identify potential leaks and anomalies, helping to diagnose the observed memory leakage.
[PLAN] Remaining Tasks: ['Fetch status database', 'Run memory diagnostics']
[TOOL] Invoking tool for: [Fetch status database]


[REFLECT] Reflection: Another step is required, as the task queue still has a pending task to "Run memory diagnostics", indicating that the current step was only preparatory and further analysis is needed.

--- [STEP 2] ---
[OBSERVE] Last Observation: Tool [Fetch status database] execution successfully verified.


[REASON] Thought: The significance of the next task, "Run memory diagnostics", is to identify and investigate potential memory leakage anomalies by analyzing system performance metrics.
[PLAN] Remaining Tasks: ['Run memory diagnostics']
[TOOL] Invoking tool for: [Run memory diagnostics]


[REFLECT] Reflection: Another step is required, as the observation verifies successful tool execution but does not directly address diagnosing memory leakage anomalies.


## Output Explanation & Complexity Analysis

- **Agent Execution Cycle**: Demonstrates how each stage of the `Observe -> Reason -> Plan -> Tool -> Reflect` loop progresses dynamically, leveraging local LLM queries at each reasoning block.
- **Time Complexity**: $O(T \cdot (P \cdot d + d^2))$ operations per iteration step.
- **Space Complexity**: $O(N_s + N_h)$ RAM capacity to maintain the session state.
- **Component Denotations**:
  - $T$: Execution loop iteration steps ($T = 2$ iterations completed).
  - $P$: Number of inputs/parameters checked.
  - $d$: Underlying model embedding dimension.
  - $N_s$: System state size in bytes.
  - $N_h$: History context logs.